# Architecture tour

GPT, LLaMA, and friends are really the same model with different component choices.
This notebook walks through the configs, compares parameter counts, and visualizes attention patterns.

In [ ]:
import mlx.core as mx
import mlx.utils

from lmxlab.analysis.attention import extract_attention_maps
from lmxlab.models.base import LanguageModel
from lmxlab.models.falcon import falcon_h1_tiny
from lmxlab.models.gpt import gpt_tiny
from lmxlab.models.llama import llama_tiny

## GPT vs LLaMA

The GPT-2 block uses LayerNorm, MHA, GELU FFN, sinusoidal positions, and bias everywhere.
LLaMA swaps in RMSNorm, GQA, SwiGLU (Shazeer, 2020), RoPE (Su et al., 2021), and drops all biases.
In lmxlab these are just different strings in the config.

In [ ]:
gpt_cfg = gpt_tiny()
llama_cfg = llama_tiny()

for name, cfg in [("GPT", gpt_cfg), ("LLaMA", llama_cfg)]:
    b = cfg.block
    print(
        f"{name}: attn={b.attention}, ffn={b.ffn}, "
        f"norm={b.norm}, pos={b.position}, "
        f"bias={b.bias}"
    )

In [ ]:
tokens = mx.zeros((1, 16), dtype=mx.int32)

models = {}
for name, cfg in [("GPT", gpt_cfg), ("LLaMA", llama_cfg)]:
    m = LanguageModel(cfg)
    mx.eval(m.parameters())
    out, _ = m(tokens)
    mx.eval(out)
    models[name] = m
    print(f"{name}: {m.count_parameters():,} params, output {out.shape}")

## MHA vs GQA

In standard MHA every head has its own K, V projections.
GQA (Ainslie et al., 2023) shares K/V across groups of query heads —
same expressiveness from the Q side, but the KV cache shrinks by the group ratio.

In [ ]:
# extract attention maps from both
short_tokens = mx.array([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=mx.int32)

gpt_maps = extract_attention_maps(models["GPT"], short_tokens)
llama_maps = extract_attention_maps(models["LLaMA"], short_tokens)

n_q = llama_cfg.block.n_heads
n_kv = llama_cfg.block.effective_n_kv_heads
n_gpt = gpt_cfg.block.n_heads
print(f"GPT (MHA):  {n_gpt} Q heads, {n_gpt} KV heads")
print(f"LLaMA (GQA): {n_q} Q heads, {n_kv} KV heads (ratio {n_q // n_kv}:1)")
print(f"\nKV cache: GQA is {n_q // n_kv}x smaller than MHA")

In [ ]:
# causal masking — position i attends only to j <= i
w = gpt_maps["layer_0"]
mx.eval(w)
print("GPT layer 0, head 0 attention weights:")
for i in range(8):
    row = [f"{w[0, 0, i, j].item():.2f}" for j in range(8)]
    print(f"  pos {i}: [{', '.join(row)}]")

In [ ]:
# parameter savings from GQA
d = 64  # d_model for tiny configs
mha_attn_params = 4 * d * d  # Q, K, V, O
gqa_kv_dim = n_kv * (d // n_q)
gqa_attn_params = 2 * d * d + 2 * d * gqa_kv_dim  # Q, O full; K, V reduced

print(f"MHA attention params: {mha_attn_params:,}")
print(f"GQA attention params: {gqa_attn_params:,}")
print(f"Savings: {(1 - gqa_attn_params / mha_attn_params) * 100:.0f}%")

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_attention_heatmap

    fig = plot_attention_heatmap(gpt_maps["layer_0"], head=0, layer=0)
    fig.suptitle("GPT MHA — layer 0, head 0")
    fig.show()
except ImportError:
    print("pip install matplotlib for plots")

## SSM/attention hybrids

Falcon-H1 (Zuo et al., 2025) interleaves Mamba-2 SSM layers with GQA in an MMM* pattern.
SSM layers run in $O(n)$ vs attention's $O(n^2)$, but can't do precise recall of distant tokens.

In [ ]:
falcon_cfg = falcon_h1_tiny()
for i in range(falcon_cfg.n_layers):
    bcfg = falcon_cfg.get_block_config(i)
    ltype = "Mamba-2" if "mamba" in bcfg.attention else "GQA"
    print(f"  layer {i}: {ltype}")

falcon = LanguageModel(falcon_cfg)
mx.eval(falcon.parameters())
models["Falcon-H1"] = falcon
print(f"\nFalcon-H1: {falcon.count_parameters():,} params")

## Parameter breakdown

In [ ]:
for name, model in models.items():
    total = model.count_parameters()
    embed = model.embed.weight.size
    blocks = sum(
        p.size
        for _, p in mlx.utils.tree_flatten(
            [b.parameters() for b in model.blocks]
        )
    )
    print(f"{name}: {total:,} total = {embed:,} embed + {blocks:,} blocks")